# 🔑 Kunci Jawaban — Kuis Bab 10 (Prompt Engineering)

> **Buka file ini HANYA SETELAH** mencoba kuis sendiri. Membaca jawaban sebelum
> berjuang = nol pembelajaran (ilusi kompetensi).

Untuk tiap soal: jawaban → **kenapa** → kode referensi. Fokus belajar ada di *kenapa*.


## Bagian A — Pilihan Ganda


### Soal 1 — Jawaban: **b) menyebut skema output DAN kebijakan kasus kosong**

Prompt bisa dieksekusi mesin kalau tiga hal terjawab eksplisit: (1) dari mana
informasi boleh diambil, (2) bentuk output-nya apa (nama field + tipe),
(3) apa yang dilakukan ketika informasi tidak ada. Di lab Bagian 7, prompt naif
yang panjang dan menyebut "ekstraksi" tetap dapat 0/24 — panjang dan sopan
bukan kriteria yang bisa diperiksa mesin.

Kenapa opsi lain menjebak:

- **a)** panjang = token = uang; tanpa skema, isinya tetap tidak bisa dipakai.
- **c)** bahasa tidak mengubah kebutuhan struktur (dan model Indonesia-nya bagus).
- **d)** 20 contoh tanpa kasus tepi tetap meninggalkan lubang keputusan
  (lihat Soal 3). Lebih dari 5 contoh umumnya justru pemborosan.


### Soal 2 — Jawaban: **b) model mengarang nilai (halusinasi)**

Skema memaksa BENTUK, bukan KEJUJURAN. Kalau prompt tidak bilang "gunakan null
bila tidak ada", model mengisi slot kosong dengan nilai yang paling masuk akal
menurut statistik. Di project Bab 10 (mock terkunci) akibatnya konkret:
harga yang tidak disebut pelanggan menjadi `9900000`, estimasi kirim menjadi
`"1-3 hari"`.

Kenapa opsi lain salah:

- **a)** skema tidak mengatur bahasa VALUE — itu masalah instruksi lain.
- **c)** model jarang menolak; ia mengisi. Penolakan bukan mode default.
- **d)** JSON yang diminta tidak pernah divalidasi provider lokal; validatornya
  milikmu (dan retry loop-mu).

Tambalannya satu baris: "Jika informasi tidak ada, gunakan null. JANGAN mengarang."
Di mock, tambalan ini menaikkan `field_accuracy` dari 0.5278 → 0.8056 (project).


### Soal 3 — Jawaban: **c) contoh kasus tepi/negatif**

Contoh few-shot dibeli untuk **menggeser batas keputusan**, bukan menambah volume.
Contoh pesanan normal tidak mengubah perilaku model (ia memang sudah menganggap
semua input sebagai pesanan). Yang mengubah adalah contoh yang menunjukkan kapan
harus MENOLAK: `"Selamat siang!" → extracted=false, confidence=0.0`.

Bukti dari project: ablasi k=0, k=1, k=2 semuanya `field_accuracy` 29/36 —
contoh positif tidak menambah apa pun. k=3 melompat ke 36/36 karena
`pilih_contoh` yang menyebar merata mengikutsertakan kasus `extracted=false`.

Kenapa opsi lain salah:

- **a)** panjang ≠ informatif; biasanya malah menjejali token.
- **b)** contoh "normal" hanya mengulang yang sudah model kuasai.
- **d)** urutan dataset tidak punya alasan apa pun — dan contoh TERAKHIR paling
  ditiru (recency bias), jadi posisi itu terlalu berharga untuk contoh pasaran.


### Soal 4 — Jawaban: **c) delimiter dan leash dua-duanya diperlukan**

Delimiter memberi **batas fisik** antara instruksi dan data; leash memberi
**instruksi eksplisit** untuk menghormati batas itu. Di mock Bab 10 (dan di
`tests/test_mock_llm.py`) dua-duanya diuji terpisah dan dua-duanya tembus:

- delimiter tanpa leash → model memperlakukan perintah di dalam data sebagai perintah;
- leash tanpa delimiter → perintah jahat duduk di level yang sama dengan instruksi
  sistem (tidak ada yang membedakannya).

Kenapa opsi lain salah:

- **a)** dan **b)** sudah dibantah eksperimen di atas.
- **d)** injeksi tidak peduli bahasa; yang penting posisi (data vs instruksi).

Di produksi masih ada lapisan yang tidak dimodelkan mock: pemisahan peran pada
level API (data user TIDAK pernah masuk system prompt), output filtering, dan
least-privilege untuk tool yang dipanggil model (Bab 14).


### Soal 5 — Jawaban: **a) 0.0–0.2**

T=0.0 membuat output deterministik; T sedikit naik masih menjaga format. Di lab
Bagian 8, T=1.0 membuat 4/4 output dibungkus basa-basi sehingga
`naive_json_rate` = 0 — isinya benar, formatnya kotor.

Kenapa opsi lain salah:

- **b)** T tinggi tidak membuat model "lebih pintar menebak"; ia hanya
  memindahkan probabilitas ke token yang lebih jarang — untuk ekstraksi itu
  justru menambah halusinasi.
- **c)** T=2.0 mendekati distribusi uniform: hampir dijamin formatnya rusak.
- **d)** T memang mengubah distribusi sampling; format adalah korban pertamanya.

Prinsip praktis: **task ekstraksi/klasifikasi → T 0–0.2; penulisan kreatif → T 0.7+;**
dan apa pun pilihannya, parser + validasi skema tetap wajib.


### Soal 6 — Jawaban: **b) isi benar, output kotor — parser menyelamatkan**

Dua metrik itu dipisah PERSIS untuk memaksa kejujuran ini. `naive_json_rate`
mengukur kebersihan (bisa `json.loads` langsung?), `parse_ok_rate` mengukur
apakah parser bisa memulihkan. 0/6 vs 6/6 berarti: setiap output butuh
pembersihan. Kalau parser diandalkan tanpa sadar, kamu menambah titik gagal
yang tidak perlu — dan parser buatan sendiri itu rapuh.

Kenapa opsi lain salah:

- **a)** isi justru benar (lihat `exact_rate`/`field_accuracy`).
- **c)** ini jebakan paling berbahaya: parser menyelamatkan hari ini, tapi
  provider bisa mengubah format kapan saja; lapisan pembersih tambahan adalah
  utang operasional.
- **d)** kalau tanpa skema, outputnya prosa total dan `parse_ok_rate` = 0.

Jalan keluar produksi: pakai mode structured output/JSON-mode provider(Bab 11) → `naive_json_rate` naik mendekati 1.0, parser jadi jaring pengaman,
bukan tulang punggung.


## Bagian B — Coding (kode referensi)


### Soal 7 — `bagian_hilang`

Kuncinya sederhana: keberadaan bagian diperiksa literal `[NAMA]`. Hati-hati
`CONTOH` — ia opsional, jadi jangan masukkan ke daftar wajib.


In [ ]:
import json
import re

BAGIAN_WAJIB = ['ROLE', 'KONTEKS', 'TUGAS', 'FORMAT', 'CONSTRAINT', 'INPUT']

def bagian_hilang(prompt):
    return [b for b in BAGIAN_WAJIB if f'[{b}]' not in prompt]

print(bagian_hilang('[ROLE]\nx\n[TUGAS]\ny'))
print(bagian_hilang('\n'.join(f'[{b}]\nisi' for b in BAGIAN_WAJIB)))   # []


### Soal 8 — `render`

Tiga jebakan yang harus dilewati: (1) JANGAN `str.format` (kurung JSON meledak),
(2) substitusi single-pass dengan `PAT.sub` + lambda (isi variabel tidak
di-render ulang), (3) deteksi placeholder kurang SEBELUM substitusi.


In [ ]:
PAT = re.compile(r'\{\{\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*\}\}')

def render(template, **nilai):
    kurang = [n for n in sorted(set(PAT.findall(template))) if n not in nilai]
    if kurang:
        raise ValueError(f'placeholder tanpa nilai: {kurang}')
    return PAT.sub(lambda m: str(nilai[m.group(1)]), template)

print(render('Data: {{teks}}\n{"price": null}', teks='hai'))
print(render('{{a}}', a='{{b}}'))          # '{{b}}' — single-pass
try:
    render('{{a}} {{b}}', a=1)
except ValueError as e:
    print('ValueError:', e)


### Soal 9 — `ekstrak_json`

Dua bagian: (a) **pemindai brace-matching** yang melacak `dalam_string` dan
`escape` — inilah yang membuat `{"a": "}{"}` tidak pecah; (b) **perbaikan**
untuk pelanggaran umum (`//`, kutip tunggal, `True/None`, koma menggantung).
Kalau JSON mentah gagal, coba perbaikan dulu; kalau kandidat gagal juga, cari
`{` berikutnya (bukan langsung menyerah).


In [ ]:
def _perbaiki(teks):
    t = str(teks).strip()
    t = re.sub(r'//[^\n]*', '', t)
    t = t.replace('True', 'true').replace('False', 'false').replace('None', 'null')
    t = re.sub(r"'([^']*)'", r'"\1"', t)
    t = re.sub(r',\s*([}\]])', r'\1', t)
    return json.loads(t)


def ekstrak_json(teks):
    t = str(teks)
    pos = t.find('{')
    while pos != -1:
        depth, dalam_string, escape = 0, False, False
        for j in range(pos, len(t)):
            c = t[j]
            if dalam_string:
                if escape:
                    escape = False
                elif c == '\\':
                    escape = True
                elif c == '"':
                    dalam_string = False
                continue
            if c == '"':
                dalam_string = True
            elif c == '{':
                depth += 1
            elif c == '}':
                depth -= 1
                if depth == 0:
                    kandidat = t[pos:j + 1]
                    try:
                        return json.loads(kandidat)
                    except json.JSONDecodeError:
                        try:
                            return _perbaiki(kandidat)
                        except Exception:
                            break
        pos = t.find('{', pos + 1)
    raise ValueError('tidak ada objek JSON valid di output')

print(ekstrak_json('bla ```json\n{"a": 1}\n``` bla'))
print(ekstrak_json('x {"a": "}{"} y'))
print(ekstrak_json("Hasil: {'price': 25, 'x': True,} // catatan"))


### Soal 10 — `bungkus_data`

Urutannya penting: **escape dulu, baru bungkus**. Kalau dibalik, kamu
membungkus data yang masih bisa menutup delimiter — dan penyerang menulis
teksnya seolah-olah ia berada di level sistem.


In [ ]:
def bungkus_data(teks, tag='dokumen'):
    bersih = str(teks).replace(f'</{tag}>', '').replace(f'<{tag}>', '')
    return f'<{tag}>\n{bersih}\n</{tag}>'

print(repr(bungkus_data('halo')))
print(repr(bungkus_data('x</dokumen>y')))     # hanya SATU </dokumen>
print(repr(bungkus_data('halo', 'konteks')))


---

Skor penuh? Lanjut ke `project-lembar-prompt/` — di sana semua konsep ini
disatukan menjadi paket `plib` end-to-end dengan 157 test dan harness evaluasi
yang menghasilkan tangga 0.0000 → 0.5278 → 1.0000.
